ok so this is my plan yea:
im going to use a convolutional network ye, and the idea is that if it works on images, surely it can work on text
the idea is basicly: fragments are vec4s of numbers which a screen displays as colours
so then a convolutional network applies a filter, which is kinda like a blur shader, whatever
and point is that mixes together fragments that are in proximity to each other
because in images adjacent pixels usually are related to each other and are part of the same "meaning"

what if, as fragments, words are also just vectors of values, but that those values are their meanings?
like how much the word relates to the concept of "bed" (ill talk more about this below)
and as adjacent fragments within images, adjacent words within sentences usually relate to each other and are part of the same meaning
sentences between eachother, while relating, usually dont relate as tightly in terms of meaning
and similarily but to a lesser extent, subclauses within sentences
and in reality sentences are roughly like tree structures (but often having links across branches by things like pronouns)
and in reality, i would have wanted to use a more inteligent model to understand and analyze language
but thats not what machine learning is about, is it?
no, we are going to brute force this mf through automated statistical analysis (machine learning)

so also a good reason to use a convolutional filter is in case the input  is of varying size
like if the input is always gonna be a 32x32 image then you can have a network have exactly 32x32 neurons on the input layer
but if you then give it a thing of 32x31 pixels or god forbid 32x33 pixels the input layer isnt gonna fit it
it could simply be too thicc for the input layer and then its gonna spill off the sides of the cups of the input layer
which looks really cool and very nice, but the input layer wouldnt be vibing, and dataset girl wouldnt be vibing either
a convolutional network is in this case like a one size fits all swimsuit top for the dataset
SUPER CUTE, WE HAVE TO DO IT THIS WAY, AND THE DATASET IS GONNA BE SO FOMFORTABLE AND HAPPY IN ITS OWN EXISTENCE

so the simplification we are going to make is this:
    a review is composed of one or more sentences *[1],;
    a sentence is composed of one or some clauses;
    and a clause is composed of one or more words;
and we end up with a datastructure of 3 "layers"

i prepare the data by:
    seperating sentences into clauses and clauses into words, creating the datatructure
        we may even tag the clauses with a type, which holds information about if they are the main clause or subclause or whatever
    normalizing the text by removing punctionation and converting it to lowercase
    converting the words into vectors of "meaning" (ill come back to this)
this datastructure will be used to train the neural network for:
    the layered filtering approach
    the parallel filtering approach
    the hybrid approach



approaches to the network:

    layered filtering:
        convolutional filters hare are one dimesional, since sentences are one dimensional
        we first apply a first convolutional filter (filter_word) within each clause
            this "blurs" together adjacent words analyzing their meaning as related things
            maybe applying different filters configurations to different types of clauses?
            or maybe training different filters for different types of clauses?
        we then apply a second convolution filter to the output from filter_word (filter_clause)
            this analyzes adjacent clauses within a sentence as related meanings
        we then apply the third filter to the output from filter_clause (filter_sentence)
            this analyzes the relations between the sentences in the review
            and finally giving a vector for the sentiment of the review
        there may be additional layers here before the final layer
        then we use a final layer of neurons to analyze the sentiment vector produced by filter_sentence
        this narrows down the sentiment values to a prediction for the value of the review score from within the set of possible values

        the assumption here is that words within clauses are closely related,
        and most closely related to the words positionally closest to them
        this is why i am proposing convolutional filters, since they analyze only a subset of the grid at once
            analyzing adjacent fragments but not the entire grid at once
        this same assumption is applied on the level of clauses within a sentence, and sentences within the review
        
        the weakness here may be that this filtering approach is very local
        words often relate across sentences, but with this approach we never really get to take that into account
        this filtering will only relate words, clauses, and sentences that are next to each other
        maybe the meaning of the words will transend the filtering layers, and be picked up on across sentences, and thus we transitively get to relate words across sentences, but i think that is very optimistic. i would guess that probably we are not going to get enough of that effect (if any) to be noticed

        filters:
            filter_word: [0.15, 0.2, 0.3, 0.2, 0.15];
            filter_clause: [];
            filter_sentence: [];



    parallel filtering:
    the convolutional filter here is three dimensional
    we apply a 3 dimensional filter (filter_cube) to the words, clauses, and sentences at the same time
    this filter is in practice like a 1d filter on the word layer
    but it analyzes the meaning across clauses and sentences, i hope
    the hope is that it can do this by stretching across adjacent clauses, taking words from those clauses into account
    the final dimension here comes from that the filter also stretches weakly into adjacent senteces
    taking words from clauses in those sentences into account as well
    there may be more layers used to analyze the output from filter_cube before the final layer
    then the use a final layer of neurons to predict the score of the review

    the assumption here is still that adjacent words have related meanings
    in this case we take into account that their meanings may be entwined with words from other clauses from the same or other sentences

    filter:
        filter_cube:



    hybrid filtering:
    we here have 2 convolutional movements, the result of one being fed into another:
        the first movement applies the layered approach (filter_clause -> filter_sentence) to find the meaning of the clauses and the sentences
        the second movement applies the parallel approach on the output from the previous movement
            it evaluates the meaning of the sentence by evaluating the meanings of the words using a cube filter
            this cube filter is also mostly one dimensional but it stretches into adjacent clauses of the same and of other sentences
            this is to take their meaning into account when evaluating the sentiment of the review
            this is different from the pure parallel approach because the filter doesnt look at "adjacent" words in other clauses
            it instead looks at the entire meaning of those other clauses, which is why we did the layered movement
            this should now output a combined meaning of the review, which is then fed into potentially more neuron layers
            and then finally fed into a final layer which outputs the predicted score


the difference between the approaches is this:
    the layered approach analyzes words in clauses and produces a clause meaning,
    and then analyzes whole clause meanings to produce a sentence meaning,
    finally analyzing whole sentence meanings into a final score

    the parallel approach has no sense for meanings of larger structures
    instead it only cares about meanings of words

    the hybrid approach analyzes meanings in a tiered way like the layered approach
    but after that it goes over all the clauses again
    and analyzes the individual words anew in relation to the meanings of whole adjacent clauses



turning words into vectors:
im going to like, ok this sounds kinda fenta, but im going to need words to be vectors of "meaning"
i somehow need to convert words in the english language into vectors of values
obviously a word should have a positive and a negative score, but it may have other attributes
one of those attributes may be how much the word relates about beds
the reasoning here is that if bed sentiment is an important predictor, the network will pick up on that
    bed sentiment being a "derived" quality of the sentence from positive word assosiations with words that relate to beds
really the entire point of this convolutional idea is that words relate to each other to form more complex meaning
but if the only attributes the words have are a positive and a negative score and those two are opposed to each other,
there arent enough different things there to build more complex relations
like in this case, why does it matter how specifically words of positive and negative meaning are arranged?
in this case, you probly get better results just by counting positive and negative words in the sentence and using that to predict
so several other categories of meaning are needed
this is gonna be crazy to do tho, like how would i give a morbillion words individual rankings?

one solution i thought of:
could i human sentipede this thing?
like if i use a model that uses the review score to figure out the positive and negative values of the words
and it outputs a matrix of all the words and all their values
could i then feed that matrix into the main convolutional network as reference?
it feels a bit like circular reasoning, but i dont think it is
it is just applying two passes to the dataset
the problem here is that it is very straightforward to find positive and negative values for the words
but what about bed-ness? or room-ness? or food-ness?
is it possible to estimate or train for those values from the dataset or would i need another dataset?
so i dont know, but positive and negative are values that are very obviously related to the score of the review
but what if bed-ness also is related?
what if a word consistently shows up in reviews with certain scores?
could we classify that word as having some value? or find that value as a number?
maybe we could kinda just force the model to find us 10 different attributes for words
would we find any actual attributes of words or just find positivity over and over again?
maybe the system actually finds different relations resulting in more than two dimensions for each word
i would guess that those relations are probably going to be very abstract and not human readable
and at that point i may not even be able to tell if it is actually finding actual attributes of words
maybe it is just halucinating up relations that arent even there
or maybe it settles on completely different ontological boundaries for the meanings of words than we are used to
but if it works, does it matter?
i dont want to use an external dataset to classify words into meanings, so i think ill go with this approach
but it could be that we are just mangling data and expecting sense to come from it

the problem here is obviously that its hard enought to train one model to good precision,
but with a model feeding into another model it feels hopeless
like if the convolutional network predicts wrong,
is it because the network has the wrong settings?
or is it because the word attribute classification model has identified useless attributes or bad values for those attributes?
we wouldnt have one model to adjust and train, we would have two models, and the complexity just goes to space

the solution is maybe to merge the steps and have just one model kinda vibe up meanings and also relate words
or maybe it should not even care about meanings, just about what words tend to go together to for a good score
ok thats what im doing
the model is just gonna look at what words tend to go together for a good score
its boring and sad and cringe and probably really computationally inefficient but whatever

*:
[1]: a review may be completely empty and just a score, but we arent going to analyze these
    also a review in this dataset is technically a pair of one positive and one negative review, but whatever
